<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/RAG_%ED%95%99%EC%8A%B5_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%A7%8C%EB%93%A4%EA%B8%B0_%EB%84%A4%EA%B1%B0%ED%8B%B0%EB%B8%8C_%EC%83%98%ED%94%8C_%EB%A7%8C%EB%93%A4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install FlagEmbedding peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.4 MB/s eta 0:00:00
  Created wheel for FlagEmbedding: filename=FlagEmbedding-1.3.5-py3-none-any.whl size=233746 sha256=1d9e997e895db6501cd3e76b22f8053c27028f09287cdd8d5181c2966b9c3d60
  Stored in directory: /root/.cache/pip/wheels/b2/1f/f6/78f862bb80cb959cc9960b7c4e2d1f702b1bc0e79d19b5f124
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=4440fad259ca699b87c591a68aaaa91517bca96d79961ca5d86fdeca755b9c5e
  Stored in directory: /root/.c

In [2]:
!wget https://github.com/KLUE-benchmark/KLUE/raw/main/klue_benchmark/klue-mrc-v1.1/klue-mrc-v1.1_train.json

--2026-03-17 06:02:48--  https://github.com/KLUE-benchmark/KLUE/raw/main/klue_benchmark/klue-mrc-v1.1/klue-mrc-v1.1_train.json
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/klue-mrc-v1.1/klue-mrc-v1.1_train.json [following]
--2026-03-17 06:02:48--  https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/klue-mrc-v1.1/klue-mrc-v1.1_train.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47952737 (46M) [application/octet-stream]
Saving to: ‘klue-mrc-v1.1_train.json’

klue-mrc-v1.1_train 100%[===================>]  45.73M   201MB/s    i

In [3]:
import json
import pandas as pd
from tqdm import tqdm

#데이터 불러오기

JSON 데이터셋 추출로 context-question-answer 데이터 생성

In [4]:
with open('klue-mrc-v1.1_train.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 필요한 정보 추출
records = []
for article in data['data']:
    title = article['title']
    news_category = article.get('news_category', 'N/A')
    source = article.get('source', 'N/A')

    for paragraph in article['paragraphs']:
        context = paragraph['context']

        for qa in paragraph['qas']:
            question = qa['question']
            question_type = qa['question_type']
            is_impossible = qa['is_impossible']

            for answer in qa['answers']:
                answer_text = answer['text']
                answer_start = answer['answer_start']

                records.append({
                    'title': title,
                    'news_category': news_category,
                    'source': source,
                    'context': context,
                    'question': question,
                    'question_type': question_type,
                    'is_impossible': is_impossible,
                    'answer_text': answer_text,
                    'answer_start': answer_start
                })

# DataFrame으로 변환
df = pd.DataFrame(records)

In [10]:
df = df.drop_duplicates(subset='question')
df = df.drop_duplicates(subset='context')

df = df[['context', 'question', 'answer_text']].iloc[:99]

In [11]:
len(df)

99

#네거티브 샘플 생성

문장 임베딩 함수

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-m3')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [7]:
import numpy as np

def get_embeddings(sentences, batch_size=32):
    return model.encode(sentences, batch_size=batch_size)

In [13]:
#context -> 임베딩
context_embeddings = get_embeddings(df['context'].to_list())

In [36]:
negative_samples_list = []

for i in range(len(df)):
  #질문 -> 임베딩
  question = df['question'].iloc[i]
  question_embedding = get_embeddings(question)

  #context 임베딩 x 질문 임베딩
  similarities = np.dot(context_embeddings, question_embedding)

  #코사인 유사도 높은 idx 추리기
  top_n_idxs = [idx for idx in similarities.argsort()[::-1] if idx != 1][:4]

  negative_samples = [df['context'].iloc[idx] for idx in top_n_idxs]
  negative_samples_list.append(negative_samples)


In [39]:
df['negative_samples'] = negative_samples_list
df = df[['question','context','negative_samples','answer_text']]
df.head()

,question,context,negative_samples,answer_text
0,북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?,올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정도...,[“아폴로우주선을 타고 지금까지 여섯 차례에 걸쳐 12명이 달 표면에 발을 디뎠습니...,한 달가량
2,지능형 생산자동화 기반기술을 개발중인 스타트업은?,부산시와 (재)부산정보산업진흥원(원장 이인숙)이 ‘2020~2021년 지역SW서비스...,[방송일시(인터넷 LIVE) : 2020년 12월 10일 (목) 낮 15:00 – ...,삼보테크놀로지
3,개막전에서 3안타 2실점을 기록해서 패한 선수는?,시범 경기에서는 16이닝을 던져 15실점을 기록하는 등 성적이 좋지 않았지만 본인으...,[시범 경기에서는 16이닝을 던져 15실점을 기록하는 등 성적이 좋지 않았지만 본인...,와쿠이 히데아키
4,컵라면 매출에서 불닭볶음면을 이긴 상품은?,유명 맛집 이름을 달고 나온 편의점 자체상표(PB) 라면이 인기를 끌고 있다. ‘검...,[유명 맛집 이름을 달고 나온 편의점 자체상표(PB) 라면이 인기를 끌고 있다. ‘...,‘교동반점 짬뽕’
8,정부에게 환경과 관련해서 우선적으로 원조 받고 있는 곳은?,“앞으로 5년 안에 아시아 친환경·신재생에너지 투자의 황금기가 도래할 것입니다.”기...,[“앞으로 5년 안에 아시아 친환경·신재생에너지 투자의 황금기가 도래할 것입니다.”...,환경오염이 심한 지역


In [40]:
df['negative_samples'].iloc[0]

['“아폴로우주선을 타고 지금까지 여섯 차례에 걸쳐 12명이 달 표면에 발을 디뎠습니다. 그럼 가장 깊은 바다라고 하는 마리아나 해구의 챌린저 해연(깊이 1만911±40ｍ)까지 내려간 사람은 몇 명일까요? 세 명입니다. 그것도 1960년과 2012년 두 차례뿐이죠.” 지난 18일 경기 안산시에 있는 한국해양과학기술원에서 만난 강정극 원장(사진)은 “우주에 비해 바다는 우리에게 아주 가깝고 친숙하지만 우리가 바다에 대해 아는 것은 그리 많지 않다”며 “기후 변화, 미래 식량, 광물 자원, 에너지 등의 문제는 모두 바다를 통해 해결할 수 있기 때문에 바다를 연구하는 것은 매우 중요한 일”이라고 강조했다. 한국해양과학기술원이 올해로 설립 40주년을 맞았다. 1973년 한국과학기술연구원(KIST) 부설 해양개발연구소로 설립돼 1990년 ‘한국해양연구소’로 독립했고, ‘한국해양연구원’을 거쳐 작년 7월 ‘한국해양과학기술원’으로 이름이 바뀌었다. 강 원장은 “1960년대에 존 F 케네디 미국 대통령이 취임사에서 3대 거대과학으로 우주, 원자력, 해양 개발을 꼽으면서 해양에 대한 국제적 관심이 커졌다”며 “한국은 당시 바다를 연구할 수 있는 인력이 거의 없었기 때문에 프랑스 정부와 협의해 5년 동안 매년 10명씩 연구원들을 프랑스 각 대학에 유학 보내면서 연구소가 출발했다”고 설명했다. 바다였던 해양과학기술원 앞이 시화호 매립으로 드넓은 육지로 변한 지난 40년 동안 기술원은 한국의 해양과학 연구를 이끌어 왔다. 1988년 남극 킹조지섬에 세종과학기지를 세우면서 극지방 연구를 개척한 곳도 해양과학기술원이다. 강 원장은 “극지방은 기후 변화와 지구 온난화를 가장 민감하게 반영하는 지역이기 때문에 중요성이 높은 곳”이라며 “내년에는 남극 대륙에 장보고기지도 세워져 남극 대륙에서 우주, 천문, 고층대기 분야 연구가 본격화될 것”이라고 말했다. 세종과학기지는 남극 대륙이 아닌 끄트머리에 붙은 섬에 있기 때문에 남극을 연구하는 데 한계가 있었다는 설명이다. 북극에 세워진 다산기지 